# Table de composition RCC8

La table de composition est le coeur du raisonnement RCC8. Elle permet de deduire les relations possibles entre deux regions a partir d'une region intermediaire.

Si on sait que:

- `A --R1--> B`;
- `B --R2--> C`;

alors la table donne les relations possibles pour `A --?--> C`.

In [1]:
from rcc8.composition_table import COMPOSE
from rcc8.relations import ALL_RELATIONS, inverse_relation, inverse_relations

RELATION_ORDER = ["DC", "EC", "PO", "EQ", "TPP", "NTPP", "TPPI", "NTPPI"]


def fmt(relations):
    ordered = [r for r in RELATION_ORDER if r in relations]
    return "{" + ", ".join(ordered) + "}"


assert set(RELATION_ORDER) == set(ALL_RELATIONS)

## 1. Affichage de la table

Chaque ligne correspond a `R(A,B)`, chaque colonne a `R(B,C)`, et chaque case donne les relations possibles pour `R(A,C)`.

In [2]:
def print_composition_table():
    col_width = 34
    header = "R1 \\ R2".ljust(9) + " | ".join(r.center(col_width) for r in RELATION_ORDER)
    print(header)
    print("-" * len(header))

    for left in RELATION_ORDER:
        row = [left.ljust(9)]
        for right in RELATION_ORDER:
            row.append(fmt(COMPOSE[left][right]).center(col_width))
        print(" | ".join(row))


print_composition_table()

R1 \ R2                  DC                 |                 EC                 |                 PO                 |                 EQ                 |                TPP                 |                NTPP                |                TPPI                |               NTPPI               
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
DC        | {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI} |      {DC, EC, PO, TPP, NTPP}       |      {DC, EC, PO, TPP, NTPP}       |                {DC}                |      {DC, EC, PO, TPP, NTPP}       |      {DC, EC, PO, TPP, NTPP}       |                {DC}                |                {DC}               
EC        |     {DC, EC, PO, TPPI, NTPPI}      |    {DC, EC, PO, EQ, TPP, TPPI}   

## 2. Exemples de compositions importantes

Ces exemples sont ceux que l'on retrouve souvent dans la propagation de contraintes.

In [3]:
examples = [
    ("TPP", "TPP"),
    ("NTPP", "NTPP"),
    ("TPP", "NTPP"),
    ("EC", "NTPP"),
    ("DC", "TPPI"),
    ("EQ", "PO"),
]

for left, right in examples:
    print(f"{left} o {right} = {fmt(COMPOSE[left][right])}")

assert COMPOSE["TPP"]["TPP"] == {"TPP", "NTPP"}
assert COMPOSE["NTPP"]["NTPP"] == {"NTPP"}
assert COMPOSE["EQ"]["PO"] == {"PO"}

TPP o TPP = {TPP, NTPP}
NTPP o NTPP = {NTPP}
TPP o NTPP = {NTPP}
EC o NTPP = {PO, TPP, NTPP}
DC o TPPI = {DC}
EQ o PO = {PO}


## 3. La relation `EQ` agit comme une identite

Composer avec `EQ` ne change pas la relation.

In [4]:
for relation in RELATION_ORDER:
    print(f"{relation} o EQ = {fmt(COMPOSE[relation]['EQ'])}")
    print(f"EQ o {relation} = {fmt(COMPOSE['EQ'][relation])}")
    assert COMPOSE[relation]["EQ"] == {relation}
    assert COMPOSE["EQ"][relation] == {relation}

print("EQ est bien l'identite de la composition RCC8.")

DC o EQ = {DC}
EQ o DC = {DC}
EC o EQ = {EC}
EQ o EC = {EC}
PO o EQ = {PO}
EQ o PO = {PO}
EQ o EQ = {EQ}
EQ o EQ = {EQ}
TPP o EQ = {TPP}
EQ o TPP = {TPP}
NTPP o EQ = {NTPP}
EQ o NTPP = {NTPP}
TPPI o EQ = {TPPI}
EQ o TPPI = {TPPI}
NTPPI o EQ = {NTPPI}
EQ o NTPPI = {NTPPI}
EQ est bien l'identite de la composition RCC8.


## 4. Verification de coherence par inverse

La table doit respecter la propriete suivante:

`inverse(R1 o R2) = inverse(R2) o inverse(R1)`

Cette propriete garantit que les contraintes `R(A,B)` et `R(B,A)` restent coherentes.

In [5]:
errors = []

for left in RELATION_ORDER:
    for right in RELATION_ORDER:
        expected = inverse_relations(COMPOSE[left][right])
        actual = COMPOSE[inverse_relation(right)][inverse_relation(left)]
        if expected != actual:
            errors.append((left, right, expected, actual))

print("Nombre d'erreurs:", len(errors))
assert errors == []

Nombre d'erreurs: 0


## 5. Fermeture et completude

La table doit avoir une entree pour chaque couple de relations, et chaque sortie doit rester dans RCC8.

In [6]:
for left in RELATION_ORDER:
    assert set(COMPOSE[left]) == set(RELATION_ORDER)
    for right in RELATION_ORDER:
        result = COMPOSE[left][right]
        assert result
        assert result <= set(RELATION_ORDER)

print("La table est complete et fermee sur les 8 relations RCC8.")

La table est complete et fermee sur les 8 relations RCC8.


## 6. A retenir

- La table encode les deductions indirectes RCC8.
- Les sorties sont souvent des ensembles: on deduit des relations possibles, pas toujours une relation unique.
- `EQ` agit comme identite.
- La coherence par inverse est indispensable pour maintenir `R(A,B)` et `R(B,A)` synchronises.